# Create Lemelson-MIT Awards (PRIZE PATTERN, Drupal JSON:API multi-program)

Ingest of **all 6 award programs** administered by the Lemelson-MIT Program at MIT under the Lemelson Foundation, fetched via the Drupal site's public **JSON:API** endpoint at `lemelson.mit.edu/jsonapi`. Method #2 on the runbook ladder (REST/JSON:API).

**Awarding body:** Lemelson Foundation — `F4320314845` (US). The MIT program is the administrative arm; the funder is the Lemelson Foundation that endows the prizes.

**Six programs covered in a single ingest** (verified 2026-05-22 from `/jsonapi/taxonomy_term/award`):

| Program | Rows | Years | Amount | Notes |
|---|---|---|---|---|
| Lemelson-MIT Prize | 25 | 1995-2019 | $500,000 | **Discontinued 2019** per foundation announcement |
| Lemelson-MIT Student Prize | 70 | 1995-2021 | $15,000 (grad) | 5 themed categories: Cure it / Eat it / Move it / Use it / Drive it |
| Lemelson-MIT Lifetime Achievement Award | 12 | various | NULL (§6.7 waiver) | Honorific; no published per-laureate amount |
| Lemelson-MIT Award for Sustainability | 5 | 2024+ | $100,000 | Newer program |
| Invention Apprentice | 4 | various | NULL (§6.7 waiver) | Teen invention program |
| Lemelson-MIT Award for Global Innovation | 2 | various | NULL (§6.7 waiver) | Per-laureate amount not published |
| **Total** | **118** | **1995-2021** | mix | |

**Multi-amount-tier handling**: per-row amount is assigned by program via a Python dict in the script (`AWARD_AMOUNTS`). The 3 programs without a documented per-laureate amount get NULL with a §6.7 waiver applied per-program. The §6.7 verification cell in Step 6 reports `pct_amount` per scheme so reviewers see exactly which programs are waived.

**Discovery method**: visited `/jsonapi` root → `node--award_winner` confirmed in the resource registry → fetched 3 paginated pages (29 + 47 + 42 = 118 records). Two taxonomies queried once each:
- `/jsonapi/taxonomy_term/award` → 6 program names
- `/jsonapi/taxonomy_term/prize_category` → 5 Student-Prize subcategories

Then per record: resolve `field_award` UUID → program name → amount. Per-row payload:
- `title` → recipient's full name
- `field_award_year` → award year
- `field_invention_name` → the invention/contribution
- `field_school` → home institution
- `field_location.country_code` → 2-letter ISO country
- `field_award` (rel) → which Lemelson program
- `field_prize_category` (rel) → Student-Prize subcategory (NULL for other programs)
- `body.value` → bio paragraphs (combined with invention name into `description`)

**Schema:**
- `funder_award_id` = `lemelson-mit-{program-slug}-{year}-{slug}`. Includes the program in the ID so multi-recipient years across different programs cannot collide.
- `funder_scheme` = program name (6 distinct values).
- `lead_investigator` PERSON-LEVEL: `given_name` + `family_name` parsed from `title`, `affiliation.name` populated from `field_school` (59% — Student-Prize entries are often MIT students with the school implied), `affiliation.country` populated from `field_location.country_code` (100%).
- For joint awards (e.g. "Rebecca Richards-Kortum and Maria Oden"), `is_joint=True` and the first recipient becomes `lead_investigator`; the co-laureate is captured in description.
- `funding_type = 'prize'`.
- `declined` always FALSE.

**Coverage** (verified 2026-05-22 full local fetch): 118 recipient rows / 1995-2021 / 100% description + country / 85% amount (15% waived per §6.7) / 59% affiliation.

**Source authority** — `lemelson.mit.edu` is the official site for the Lemelson Foundation's MIT program. Method #2 (REST/JSON:API) on the runbook ladder.

**Prerequisites**: Run `scripts/local/lemelson_mit_to_s3.py` first (~3 JSON:API calls + 2 taxonomy fetches, <20 sec).

**Data source**: `https://lemelson.mit.edu/jsonapi/node/award_winner` + 2 taxonomy term endpoints

**S3 location**: `s3a://openalex-ingest/awards/lemelson_mit/lemelson_mit_awards.parquet`

## Step 1: Create staging table from S3

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.lemelson_mit_raw
USING delta
AS
SELECT *, current_timestamp() AS databricks_ingested_at
FROM parquet.`s3a://openalex-ingest/awards/lemelson_mit/lemelson_mit_awards.parquet`;

In [ ]:
%sql
SELECT COUNT(*) FROM openalex.awards.lemelson_mit_raw;

## Step 1.5: Inspect raw + multi-program amount-tier coverage

All source columns from the Drupal JSON:API extraction. **Verified per-row coverage** (118 recipients, validated 2026-05-22):
- 100% `funder_award_id`, `year`, `name`, `landing_page_url`
- 100% `program`, `country`, `description`
- 85% `amount` (the 15% NULL are §6.7-waived: Lifetime Achievement, Invention Apprentice, Global Innovation — none publish a per-laureate amount)
- 59% `affiliation` (Student Prize entries often imply MIT)
- 1995-2021 year range
- 6 distinct programs

Source columns: `funder_award_id`, `year`, `slug`, `name`, `given_name`, `family_name`, `is_joint`, `program`, `category`, `invention_name`, `affiliation`, `country`, `display_name`, `description`, `amount`, `currency`, `start_date`, `end_date`, `landing_page_url`, `declined`.

In [ ]:
%sql
DESCRIBE openalex.awards.lemelson_mit_raw;

In [ ]:
%sql
SELECT * FROM openalex.awards.lemelson_mit_raw LIMIT 5;

In [ ]:
%sql
-- §1.5 multi-program amount-tier scan — confirms which programs carry an
-- amount and which are §6.7-waived. Expected:
--   Lemelson-MIT Prize:                    25 rows @ 500K
--   Lemelson-MIT Student Prize:            70 rows @ 15K
--   Lemelson-MIT Award for Sustainability:  5 rows @ 100K
--   Lifetime Achievement / Invention Apprentice / Global Innovation: NULL
SELECT program,
       COUNT(*) AS rows,
       COUNT(amount) AS has_amount,
       MIN(TRY_CAST(amount AS DOUBLE)) AS min_amount,
       MAX(TRY_CAST(amount AS DOUBLE)) AS max_amount
FROM openalex.awards.lemelson_mit_raw
GROUP BY program
ORDER BY rows DESC;

In [ ]:
%sql
-- Year × program distribution
SELECT TRY_CAST(year AS INT) AS year_int,
       program,
       COUNT(*) AS rows
FROM openalex.awards.lemelson_mit_raw
GROUP BY year_int, program
ORDER BY year_int DESC, program;

## Step 1.6: Fail-fast — verify Lemelson Foundation funder row exists

In [ ]:
%sql
-- Must return exactly 1 row.
SELECT funder_id, display_name, ror_id, doi
FROM openalex.common.funder
WHERE funder_id = 4320314845;

## Step 2: Transform to award schema (multi-program, multi-amount-tier)

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.lemelson_mit_awards
USING delta
AS
WITH funder_resolved AS (
    SELECT funder_id, display_name, ror_id, doi
    FROM openalex.common.funder
    WHERE funder_id = 4320314845  -- Lemelson Foundation
)
SELECT
    abs(xxhash64(CONCAT(
        TRY_CAST(f.funder_id AS STRING), ':', LOWER(n.funder_award_id)
    ))) % 9000000000 AS id,
    n.display_name,
    n.description,
    f.funder_id,
    n.funder_award_id,
    TRY_CAST(n.amount AS DOUBLE) AS amount,  -- variable by program; NULL for §6.7-waived programs
    n.currency,
    struct(
        CONCAT('https://openalex.org/F', TRY_CAST(f.funder_id AS STRING)) AS id,
        f.display_name,
        f.ror_id,
        f.doi
    ) AS funder,
    'prize' AS funding_type,
    n.program AS funder_scheme,  -- one of 6 distinct Lemelson programs
    'lemelson_mit' AS provenance,
    TRY_TO_DATE(n.start_date, 'yyyy-MM-dd') AS start_date,
    TRY_TO_DATE(n.end_date,   'yyyy-MM-dd') AS end_date,
    TRY_CAST(SUBSTRING(n.start_date, 1, 4) AS INT) AS start_year,
    TRY_CAST(SUBSTRING(n.end_date,   1, 4) AS INT) AS end_year,
    -- lead_investigator: PERSON-LEVEL. affiliation populated from field_school
    -- (59% — Student-Prize entries often have NULL school since MIT is
    -- implied; can be enriched later). affiliation.country populated from
    -- field_location.country_code (100%).
    CASE
        WHEN n.name IS NULL OR n.name = '' THEN NULL
        ELSE struct(
            n.given_name AS given_name,
            n.family_name AS family_name,
            CAST(NULL AS STRING) AS orcid,
            TRY_TO_DATE(n.start_date, 'yyyy-MM-dd') AS role_start,
            struct(
                n.affiliation AS name,
                n.country AS country,
                CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>) AS ids
            ) AS affiliation
        )
    END AS lead_investigator,
    CAST(NULL AS STRUCT<
        given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
        affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
    >) AS co_lead_investigator,
    CAST(NULL AS ARRAY<STRUCT<
        given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
        affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
    >>) AS investigators,
    n.landing_page_url AS landing_page_url,
    CAST(NULL AS STRING) AS doi,
    CONCAT('https://api.openalex.org/works?filter=awards.id:G',
           TRY_CAST(abs(xxhash64(CONCAT(
               TRY_CAST(f.funder_id AS STRING), ':', LOWER(n.funder_award_id)
           ))) % 9000000000 AS STRING)) AS works_api_url,
    n.declined,
    current_timestamp() AS created_date,
    current_timestamp() AS updated_date
FROM openalex.awards.lemelson_mit_raw n
CROSS JOIN funder_resolved f
WHERE n.funder_award_id IS NOT NULL
  AND n.name IS NOT NULL;

## Step 3: Insert into openalex_awards_raw at priority 103

In [ ]:
%sql
DELETE FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'lemelson_mit' AND priority = 103;

INSERT INTO openalex.awards.openalex_awards_raw
SELECT
    id, display_name, description, funder_id, funder_award_id,
    amount, currency, funder, funding_type, funder_scheme, provenance,
    start_date, end_date, start_year, end_year,
    lead_investigator, co_lead_investigator, investigators,
    landing_page_url, doi, works_api_url,
    created_date, updated_date,
    103 AS priority  -- Lemelson-MIT priority (matches CreateAwards.ipynb registry)
FROM openalex.awards.lemelson_mit_awards;

## Step 6: Verification

Full §6.1-6.7. **§6.7 amount-coverage WAIVED for 3 of 6 programs** (Lifetime Achievement, Invention Apprentice, Global Innovation — none publish per-laureate amounts). The other 3 programs (Prize, Student Prize, Sustainability) carry their documented amounts. Verified expectations from the 2026-05-22 extraction:
- Row count: **118**
- Amount-bearing programs: Prize 25 @ $500K, Student Prize 70 @ $15K, Sustainability 5 @ $100K = 100 rows with amount (85%)
- Amount-waived programs: Lifetime Achievement 12, Invention Apprentice 4, Global Innovation 2 = 18 rows NULL (15%)
- 6 distinct `funder_scheme`, 1 distinct `funding_type` ('prize')
- 1995-2021 year range
- 100% `country`, 100% `description`, 59% `affiliation`
- `declined = FALSE` everywhere

In [ ]:
%sql
SELECT COUNT(*) AS total_lemelson_award_rows FROM openalex.awards.lemelson_mit_awards;

In [ ]:
%sql
DESCRIBE openalex.awards.lemelson_mit_awards;

In [ ]:
%sql
-- §6.3 Data completeness
SELECT
    COUNT(*) AS total,
    COUNT(display_name) AS has_title,
    COUNT(description) AS has_description,
    COUNT(amount) AS has_amount,
    COUNT(start_date) AS has_start_date,
    COUNT(lead_investigator) AS has_pi,
    ROUND(try_divide(COUNT(display_name), COUNT(*)) * 100.0, 1) AS pct_title,
    ROUND(try_divide(COUNT(start_date), COUNT(*)) * 100.0, 1) AS pct_dates,
    ROUND(try_divide(COUNT(amount), COUNT(*)) * 100.0, 1) AS pct_amount,
    ROUND(try_divide(COUNT(description), COUNT(*)) * 100.0, 1) AS pct_description
FROM openalex.awards.lemelson_mit_awards;

In [ ]:
%sql
-- §6.7 amount + currency — partial-waiver report by scheme.
-- 3 of 6 schemes are §6.7-waived (no published per-laureate amount).
SELECT funder_scheme,
       COUNT(*) AS total,
       COUNT(amount) AS has_amount,
       ROUND(try_divide(COUNT(amount), COUNT(*)) * 100.0, 1) AS pct_amount,
       ROUND(MIN(amount), 0) AS min_amount,
       ROUND(MAX(amount), 0) AS max_amount,
       collect_set(currency) AS currencies
FROM openalex.awards.lemelson_mit_awards
GROUP BY funder_scheme
ORDER BY total DESC;

In [ ]:
%sql
-- Year × scheme distribution
SELECT start_year,
       funder_scheme,
       COUNT(*) AS rows
FROM openalex.awards.lemelson_mit_awards
GROUP BY start_year, funder_scheme
ORDER BY start_year DESC, funder_scheme;

In [ ]:
%sql
-- Country distribution (100% populated from field_location)
SELECT lead_investigator.affiliation.country AS country,
       COUNT(*) AS rows
FROM openalex.awards.lemelson_mit_awards
GROUP BY country
ORDER BY rows DESC
LIMIT 12;

In [ ]:
%sql
-- Sample notable Lemelson-MIT Prize winners
SELECT id, SUBSTRING(display_name, 1, 80) AS title,
       lead_investigator.given_name AS given,
       lead_investigator.family_name AS family,
       SUBSTRING(lead_investigator.affiliation.name, 1, 40) AS affiliation,
       amount, start_year
FROM openalex.awards.lemelson_mit_awards
WHERE funder_scheme = 'Lemelson-MIT Prize'
ORDER BY start_year DESC
LIMIT 10;

In [ ]:
%sql
-- Funder struct populated correctly?
SELECT funder.id, funder.display_name, funder.ror_id, funder.doi, COUNT(*) AS rows
FROM openalex.awards.lemelson_mit_awards
GROUP BY funder.id, funder.display_name, funder.ror_id, funder.doi;

In [ ]:
%sql
-- Declined must be FALSE
SELECT declined, COUNT(*) AS rows
FROM openalex.awards.lemelson_mit_awards
GROUP BY declined;